# ZeroCap fixed-test evaluation trên Google Colab

Notebook này chỉ đánh giá file `predictions.json` đã sinh cho 104 ảnh của `fixed_test_round_001`; notebook không chạy lại ZeroCap inference. Kết quả được chuẩn hóa cùng cấu trúc report của ClipCap với CIDEr, BLEU-4, CLIPScore và RefCLIPScore.

Nếu chạy trong phiên Colab mới, hãy đặt file prediction tại `MyDrive/clipcap_colab/zerocap/predictions.json`. Nếu vừa chạy ZeroCap trong cùng runtime và file còn ở `outputs/zerocap/predictions.json`, notebook có thể dùng trực tiếp file đó.

## 1. Mount Drive, clone repository và đặt đường dẫn

In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/HnhanBk415/zfs-clip-image-captioning.git'
BRANCH = 'refactor/huuthien/evaluation'
PROJECT_ROOT = Path('/content/zfs-clip-image-captioning')

if (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', BRANCH],
        check=True,
    )
elif PROJECT_ROOT.exists():
    raise RuntimeError(
        f'{PROJECT_ROOT} đã tồn tại nhưng không phải Git repository'
    )
else:
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_ROOT)],
        check=True,
    )

DRIVE_ROOT = Path('/content/drive/MyDrive/clipcap_colab')
DRIVE_PREDICTIONS_PATH = DRIVE_ROOT / 'zerocap' / 'predictions.json'
LOCAL_PREDICTIONS_PATH = PROJECT_ROOT / 'outputs' / 'zerocap' / 'predictions.json'
FEATURE_CACHE_PATH = DRIVE_ROOT / 'data_cache' / 'features' / 'clip_features.pt'
METRICS_OUTPUT_BASE = DRIVE_ROOT / 'metrics'
REPORT_OUTPUT_BASE = DRIVE_ROOT / 'reports'
RUN_TAG = 'zerocap_fixed_test_round_001_report'

if DRIVE_PREDICTIONS_PATH.is_file():
    PREDICTIONS_PATH = DRIVE_PREDICTIONS_PATH
elif LOCAL_PREDICTIONS_PATH.is_file():
    PREDICTIONS_PATH = LOCAL_PREDICTIONS_PATH
else:
    raise FileNotFoundError(
        'Không tìm thấy predictions.json. Hãy tải file lên Drive tại '
        f'{DRIVE_PREDICTIONS_PATH}'
    )

for directory in (METRICS_OUTPUT_BASE, REPORT_OUTPUT_BASE):
    directory.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Working directory: {Path.cwd()}')
print(f'Git branch: {BRANCH}')
print(f'Predictions: {PREDICTIONS_PATH}')
print(f'Feature cache: {FEATURE_CACHE_PATH}')
print(f'Run tag: {RUN_TAG}')

## 2. Cài và kiểm tra môi trường

In [ ]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'],
        check=True,
    )
if importlib.util.find_spec('pycocoevalcap') is None:
    raise RuntimeError('Không cài được pycocoevalcap')
if shutil.which('java') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(
        ['apt-get', 'install', '-y', '-qq', 'openjdk-17-jre-headless'],
        check=True,
    )
if shutil.which('java') is None:
    raise RuntimeError('Java không có trong PATH; chưa thể tính CIDEr/BLEU-4')

subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['java', '-version'], check=True)

import torch

if not torch.cuda.is_available():
    raise RuntimeError('Notebook cần Colab GPU để tính CLIPScore hiệu quả')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.get_device_name(0)}')

## 3. Xác minh predictions, fixed test và feature cache

In [ ]:
from src.clipcap.inference.features import load_feature_cache
from src.config.clipcap_config import CLIP_MODEL_NAME
from src.config.common_config import SPLIT_DIR

MANIFEST_PATH = SPLIT_DIR / 'fixed_test_round_001.json'
REFERENCES_PATH = SPLIT_DIR / 'test.json'
for path in (MANIFEST_PATH, REFERENCES_PATH, FEATURE_CACHE_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy input: {path}')

with MANIFEST_PATH.open('r', encoding='utf-8') as file:
    fixed_test_ids = json.load(file)
with REFERENCES_PATH.open('r', encoding='utf-8') as file:
    test_references = json.load(file)
with PREDICTIONS_PATH.open('r', encoding='utf-8') as file:
    prediction_records = json.load(file)

if not isinstance(prediction_records, list):
    raise TypeError('ZeroCap predictions.json phải chứa một JSON list')
if len(fixed_test_ids) != 104 or len(set(fixed_test_ids)) != 104:
    raise ValueError('fixed_test_round_001 phải chứa đúng 104 image ID duy nhất')
if len(prediction_records) != 104:
    raise ValueError(
        f'ZeroCap predictions phải có 104 records, nhận được {len(prediction_records)}'
    )

predictions_by_id = {}
for index, record in enumerate(prediction_records):
    if not isinstance(record, dict):
        raise TypeError(f'Prediction tại index {index} không phải JSON object')
    image_id = record.get('image_id')
    caption = record.get('caption')
    if not isinstance(image_id, str) or not image_id.strip():
        raise ValueError(f'image_id không hợp lệ tại index {index}')
    if not isinstance(caption, str) or not caption.strip():
        raise ValueError(f'caption không hợp lệ tại index {index}')
    if image_id in predictions_by_id:
        raise ValueError(f'image_id bị trùng: {image_id}')
    predictions_by_id[image_id] = caption.strip()

expected_ids = set(fixed_test_ids)
actual_ids = set(predictions_by_id)
missing = expected_ids - actual_ids
extra = actual_ids - expected_ids
if missing or extra:
    raise ValueError(f'Prediction coverage: missing={len(missing)}, extra={len(extra)}')
if any(len(test_references[image_id]) != 5 for image_id in fixed_test_ids):
    raise ValueError('Mỗi ảnh fixed test phải có đúng năm reference')

cached_features = load_feature_cache(FEATURE_CACHE_PATH, fixed_test_ids)
if len(cached_features) != 104:
    raise ValueError('Feature cache không phủ đủ 104 ảnh fixed test')
del cached_features
print('Predictions: 104/104')
print('References: 5 captions/image')
print('Feature cache: 104/104')

## 4. Tạo metadata tái lập và bản input đánh giá

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def unique_prediction_value(field: str):
    values = {record.get(field) for record in prediction_records}
    if len(values) != 1:
        raise ValueError(f'Prediction metadata không đồng nhất tại {field}')
    value = values.pop()
    if value in (None, ''):
        raise ValueError(f'Prediction metadata thiếu {field}')
    return value


METRICS_OUTPUT_DIR = METRICS_OUTPUT_BASE / 'test' / RUN_TAG
EVALUATION_INPUT_DIR = METRICS_OUTPUT_DIR / 'input'
EVALUATION_INPUT_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_PREDICTIONS_PATH = EVALUATION_INPUT_DIR / 'predictions.json'
shutil.copy2(PREDICTIONS_PATH, EVALUATION_PREDICTIONS_PATH)

run_config = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'model': 'zerocap',
    'dataset': 'fixed_test_round_001',
    'num_images': 104,
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'prediction_sha256': sha256_file(EVALUATION_PREDICTIONS_PATH),
    'gpt_model': unique_prediction_value('gpt_model'),
    'clip_model': unique_prediction_value('clip_model'),
    'config_hash': unique_prediction_value('config_hash'),
    'git_commit': unique_prediction_value('git_commit'),
}
if run_config['clip_model'] != CLIP_MODEL_NAME:
    raise ValueError(
        f"CLIP model không khớp: {run_config['clip_model']} != {CLIP_MODEL_NAME}"
    )
RUN_CONFIG_PATH = METRICS_OUTPUT_DIR / 'run_config.json'
with RUN_CONFIG_PATH.open('w', encoding='utf-8') as file:
    json.dump(run_config, file, ensure_ascii=False, indent=2)
print(json.dumps(run_config, ensure_ascii=False, indent=2))

## 5. Tính CIDEr, BLEU-4, CLIPScore và RefCLIPScore

In [ ]:
metrics_command = [
    sys.executable,
    '-m',
    'src.common.caption_metrics',
    '--inference-manifest',
    str(MANIFEST_PATH),
    '--references',
    str(REFERENCES_PATH),
    '--prediction',
    f'zerocap={EVALUATION_PREDICTIONS_PATH}',
    '--feature-cache',
    str(FEATURE_CACHE_PATH),
    '--output-dir',
    str(METRICS_OUTPUT_DIR),
    '--run-config',
    str(RUN_CONFIG_PATH),
    '--clip-batch-size',
    '32',
    '--device',
    'cuda',
    '--references-per-image',
    '5',
]
print('Bắt đầu tính metric ZeroCap...')
subprocess.run(metrics_command, cwd=PROJECT_ROOT, check=True)
print(f'Metric hoàn tất: {METRICS_OUTPUT_DIR}')

## 6. Chuẩn hóa report giống ClipCap

In [ ]:
import pandas as pd
from IPython.display import display

REPORT_DIR = REPORT_OUTPUT_BASE / 'fixed_test_round_001' / RUN_TAG
REPORT_CAPTIONS_DIR = REPORT_DIR / 'captions'
REPORT_PREDICTIONS_DIR = REPORT_DIR / 'predictions'
REPORT_METRICS_DIR = REPORT_DIR / 'metrics'
for directory in (
    REPORT_CAPTIONS_DIR,
    REPORT_PREDICTIONS_DIR,
    REPORT_METRICS_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

comparison_path = METRICS_OUTPUT_DIR / 'comparison.json'
per_image_path = METRICS_OUTPUT_DIR / 'per_image_comparison.csv'
for source_path in (comparison_path, per_image_path, RUN_CONFIG_PATH):
    if not source_path.is_file():
        raise FileNotFoundError(f'Thiếu output đánh giá: {source_path}')
with comparison_path.open('r', encoding='utf-8') as file:
    comparison = json.load(file)

results_by_experiment = {
    row['experiment']: row for row in comparison['results']
}
if 'zerocap' not in results_by_experiment:
    raise ValueError('comparison.json không chứa kết quả ZeroCap')
result = results_by_experiment['zerocap']
prediction_info = comparison['prediction_files']['zerocap']
metric_artifact = {
    'generated_at_utc': comparison['generated_at_utc'],
    'experiment': 'zerocap',
    'metric_scales': comparison['metric_scales'],
    'metrics': {
        'CIDEr': result['CIDEr'],
        'BLEU-4': result['BLEU-4'],
        'CLIPScore': result['CLIPScore'],
        'RefCLIPScore': result['RefCLIPScore'],
    },
    'coverage': comparison['coverage']['zerocap'],
    'prediction_file': prediction_info,
    'inference_manifest_path': comparison['inference_manifest_path'],
    'inference_manifest_sha256': comparison['inference_manifest_sha256'],
    'references_path': comparison['references_path'],
    'references_sha256': comparison['references_sha256'],
    'feature_cache_path': comparison['feature_cache_path'],
}

with (REPORT_CAPTIONS_DIR / 'zerocap.json').open('w', encoding='utf-8') as file:
    json.dump(predictions_by_id, file, ensure_ascii=False, indent=2)
shutil.copy2(PREDICTIONS_PATH, REPORT_PREDICTIONS_DIR / 'zerocap.json')
with (REPORT_METRICS_DIR / 'zerocap.json').open('w', encoding='utf-8') as file:
    json.dump(metric_artifact, file, ensure_ascii=False, indent=2)
for source_path in (comparison_path, per_image_path, RUN_CONFIG_PATH):
    shutil.copy2(source_path, REPORT_DIR / source_path.name)

results_table = pd.DataFrame(comparison['results']).sort_values('rank')
results_table_path = REPORT_DIR / 'results_table.csv'
results_table.to_csv(results_table_path, index=False, encoding='utf-8')
archive_path = shutil.make_archive(
    str(REPORT_DIR),
    'zip',
    root_dir=REPORT_DIR,
)
display(results_table[[
    'rank',
    'experiment',
    'CIDEr',
    'BLEU-4',
    'CLIPScore',
    'RefCLIPScore',
]])
print(f'Report directory: {REPORT_DIR}')
print(f'Report archive: {archive_path}')
print('ZeroCap fixed-test evaluation đã hoàn tất')

## Cấu trúc kết quả trên Drive

```text
clipcap_colab/reports/fixed_test_round_001/
├── clipcap_fixed_test_round_001_report/
└── zerocap_fixed_test_round_001_report/
    ├── captions/
    │   └── zerocap.json
    ├── metrics/
    │   └── zerocap.json
    ├── predictions/
    │   └── zerocap.json
    ├── comparison.json
    ├── per_image_comparison.csv
    ├── results_table.csv
    └── run_config.json
```

File `zerocap_fixed_test_round_001_report.zip` được tạo bên cạnh thư mục ZeroCap report.